In [ ]:
# =============================================================================
# jobs/21_fgs_constituency_exposure.py
# How often has each constituency been forecast at each risk level?
#
# One row per constituency, calendar month and flood source, counting the days
# the forecast sat at each risk rank. It answers "which constituencies are
# repeatedly at risk" and "is this winter unusual" without scanning the
# statement history each time.
#
# WHAT COUNTS AS A DAY. Statements overlap: each looks five days ahead, so one
# date is forecast up to five times. To count each date once, this notebook uses
# DAY 1 OF THE LAST STATEMENT ISSUED ON EACH DATE (an amendment on the same day
# replaces the morning issue). Day 1 is the forecast closest to the day itself.
# Days 2 to 5 are not counted.
#
# What this script does:
#   1. Takes every statement whose intersections are complete for
#      FGS_GEOMETRY_SOURCE, and keeps the last one issued on each date.
#   2. Reads its day 1 rows from fgs_constituency_intersections (polygons
#      covering less than FGS_CONTACT_MIN_PCT of the constituency are ignored,
#      as in 14) and takes the worst rank per constituency, date and source.
#   3. Adds a source = 'any' row set: the worst across all sources.
#   4. Counts days per rank by month, and records days_covered: how many days in
#      that month have a statement at all, so a share can be worked out.
#   5. Overwrites fgs_constituency_exposure in full.
#
# SPARSE ON PURPOSE. A constituency with no flagged day in a month has no row for
# it. Use days_covered for the denominator. Dates with no statement (a gap in the
# history) are not in days_covered, so they do not look like quiet days.
#
# The current month includes today's day 1 forecast, which a later amendment can
# still change. The table is rebuilt every day.
#
# Ranks: 1 Very Low, 2 Low, 3 Medium, 4 High.
#
# NO SPATIAL WORK HERE. Plain Spark SQL, one pass over the intersections.
#
# Exit values: "success".
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

from config import (
    TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_CONST_INTERSECT,
    TBL_FGS_EXPOSURE, FGS_CONTACT_MIN_PCT, FGS_GEOMETRY_SOURCE
)
from utils.helpers import (
    get_spark, utc_now, table_exists, source_predicate, risk_rank_sql, risk_level_sql
)


# =============================================================================
# SETUP
# =============================================================================

spark   = get_spark()
now_iso = utc_now().isoformat()

for table in (TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_CONST_INTERSECT):
    if not table_exists(spark, table):
        raise Exception(f"{table} does not exist. Run 01/02 and 07_compute_intersections first.")

print(f"FGS_GEOMETRY_SOURCE = {FGS_GEOMETRY_SOURCE}")


In [ ]:
# =============================================================================
# BUILD THE TABLE
# =============================================================================

l_in_source = source_predicate(spark, TBL_FGS_INTERSECT_LOG, FGS_GEOMETRY_SOURCE, alias="l")
i_in_source = source_predicate(spark, TBL_FGS_CONST_INTERSECT, FGS_GEOMETRY_SOURCE, alias="i")

query = f"""
WITH ranked AS (
    -- complete statements, newest first within each issue date
    SELECT s.statement_id,
           to_date(s.issued_at) AS issue_date,
           ROW_NUMBER() OVER (
               PARTITION BY to_date(s.issued_at)
               ORDER BY s.issued_at DESC, s.statement_id DESC
           ) AS rn
    FROM {TBL_FGS_INTERSECT_LOG} l
    JOIN {TBL_STATEMENTS} s
      ON s.statement_id = l.statement_id
     AND COALESCE(s.last_modified_at, '') = COALESCE(l.last_modified_at, '')
    WHERE {l_in_source}
),
final AS (
    SELECT statement_id, issue_date FROM ranked WHERE rn = 1
),
day1 AS (
    -- worst rank per constituency, date and source, from day 1 of the final statement
    SELECT i.statement_id,
           i.constituency_id,
           MAX(i.constituency_name)                 AS constituency_name,
           to_date(i.forecast_date)                 AS d,
           i.source,
           MAX({risk_rank_sql('i.risk_level')})     AS rk
    FROM {TBL_FGS_CONST_INTERSECT} i
    JOIN final f ON f.statement_id = i.statement_id
    WHERE {i_in_source}
      AND i.day_index = 1
      AND i.forecast_date IS NOT NULL
      AND (i.intersection_pct IS NULL OR i.intersection_pct >= {FGS_CONTACT_MIN_PCT})
    GROUP BY i.statement_id, i.constituency_id, to_date(i.forecast_date), i.source
),
covered AS (
    -- the date each final statement's day 1 falls on; a quiet statement (no rows)
    -- falls back to its issue date
    SELECT f.statement_id, COALESCE(MIN(d.d), f.issue_date) AS d1
    FROM final f LEFT JOIN day1 d ON d.statement_id = f.statement_id
    GROUP BY f.statement_id, f.issue_date
),
cover AS (
    SELECT date_format(d1, 'yyyy-MM') AS month, COUNT(DISTINCT d1) AS days_covered
    FROM covered GROUP BY date_format(d1, 'yyyy-MM')
),
cells AS (
    SELECT constituency_id, constituency_name, d, source, rk FROM day1
    UNION ALL
    SELECT constituency_id, MAX(constituency_name), d, 'any', MAX(rk)
    FROM day1 GROUP BY constituency_id, d
)
SELECT
    c.constituency_id,
    MAX(c.constituency_name)                                        AS constituency_name,
    date_format(c.d, 'yyyy-MM')                                     AS month,
    c.source,
    SUM(CASE WHEN c.rk = 1 THEN 1 ELSE 0 END)                       AS days_very_low,
    SUM(CASE WHEN c.rk = 2 THEN 1 ELSE 0 END)                       AS days_low,
    SUM(CASE WHEN c.rk = 3 THEN 1 ELSE 0 END)                       AS days_medium,
    SUM(CASE WHEN c.rk = 4 THEN 1 ELSE 0 END)                       AS days_high,
    SUM(CASE WHEN c.rk >= 3 THEN 1 ELSE 0 END)                      AS days_medium_or_above,
    SUM(CASE WHEN c.rk >= 1 THEN 1 ELSE 0 END)                      AS days_flagged,
    MAX(c.rk)                                                       AS worst_rank,
    {risk_level_sql('MAX(c.rk)')}                                   AS worst_level,
    MAX(cv.days_covered)                                            AS days_covered,
    '{now_iso}'                                                     AS computed_at
FROM cells c
JOIN cover cv ON cv.month = date_format(c.d, 'yyyy-MM')
WHERE c.rk >= 1
GROUP BY c.constituency_id, date_format(c.d, 'yyyy-MM'), c.source
"""

result = spark.sql(query).cache()
n_rows = result.count()

n_days = spark.sql(f"""
    SELECT COUNT(DISTINCT issue_date) FROM (
        SELECT to_date(s.issued_at) AS issue_date
        FROM {TBL_FGS_INTERSECT_LOG} l
        JOIN {TBL_STATEMENTS} s
          ON s.statement_id = l.statement_id
         AND COALESCE(s.last_modified_at, '') = COALESCE(l.last_modified_at, '')
        WHERE {l_in_source}
    )
""").collect()[0][0]

print(f"Dates with a complete statement: {n_days}")
print(f"Rows (constituency x month x source): {n_rows}")

if n_days == 0:
    raise Exception(
        f"No statement has up-to-date intersections for geometry source "
        f"'{FGS_GEOMETRY_SOURCE}'. Run 07_compute_intersections first."
    )
if n_rows == 0:
    print("No constituency was forecast at any risk on day 1 of any statement. The table will be empty.")


In [ ]:
# =============================================================================
# WRITE TO DELTA
# Full overwrite: cheap to rebuild, and an amended statement changes its day.
# =============================================================================

(
    result.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TBL_FGS_EXPOSURE)
)

spark.sql(f"""
    COMMENT ON TABLE {TBL_FGS_EXPOSURE} IS
    'How many days each constituency was forecast at each FGS risk rank, by calendar
     month and flood source (source = any is the worst across sources). A day is day 1
     of the last statement issued on that date, so each date counts once. Sparse:
     months with no flagged day have no row; use days_covered (days in that month with
     a statement) as the denominator. Rebuilt daily by 21_fgs_constituency_exposure.
     Ranks: 1 Very Low, 2 Low, 3 Medium, 4 High.'
""")

written = spark.table(TBL_FGS_EXPOSURE).count()
print(f"\nVerified: {written} rows written to {TBL_FGS_EXPOSURE}.")
result.unpersist()

dbutils.notebook.exit("success")
